# 动手练习：基因调控网络推断与共表达分析

来自《生物医学网络科学》（Biomedical Network Science）。 参考答案版。

## 任务 1：选软阈值幂次 β

模拟数据：60 个样本、120 个“基因”。其中 4 个共表达模块（30/25/20/15 个基因）分别由 4 个潜在因子驱动，另有 30 个纯噪声基因；模块 1 的潜在因子与一个“临床性状”相关。数据只用于演示方法，不代表任何真实基因。

对 $\beta=1,\dots,12$ 计算无符号邻接 $A=|R|^{\beta}$，用 `scale_free_fit(A)` 得到无标度拟合指数（$\log_{10}p(k)$ 对 $\log_{10}k$ 回归的 $R^2$ 乘以斜率的负号）和平均连接度，按 [WGCNA](gene-network-inference.wgcna.html) 的准则选出拟合指数**首次 ≥ 0.85** 的最小 $\beta$。

要得到的变量：`fits`（长度 12 的列表，依次对应 β = 1…12）、`beta_star`（整数）。

期望看到：β 增大时拟合指数先快速上升，然后在高位波动；平均连接度单调下降。

> **提示：** `np.abs(R) ** b`；用 `next(b for b, f in zip(betas, fits) if f >= 0.85)` 找第一个满足条件的 β。

In [ ]:
# 准备（直接运行）
import numpy as np
from scipy import stats
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import matplotlib.pyplot as plt

def simulate(seed=13):
    """返回表达矩阵 X（样本 × 基因）、真实模块标签（0 为噪声）、真实载荷、临床性状。"""
    rng = np.random.default_rng(seed)
    n, sizes = 60, [30, 25, 20, 15]
    trait = rng.normal(size=n)
    factors = [0.8 * trait + 0.6 * rng.normal(size=n)] + [rng.normal(size=n) for _ in sizes[1:]]
    X, labels, load = [], [], []
    for m, (s, f) in enumerate(zip(sizes, factors)):
        for l in rng.uniform(0.5, 0.95, size=s):          # 载荷越大，越像模块“核心”
            X.append(l * f + np.sqrt(1 - l ** 2) * rng.normal(size=n))
            labels.append(m + 1); load.append(l)
    for _ in range(30):
        X.append(rng.normal(size=n)); labels.append(0); load.append(0.0)
    return np.array(X).T, np.array(labels), np.array(load), trait

def scale_free_fit(A, nbins=10):
    k = A.sum(axis=1) - 1.0                                 # 减去对角线 a_ii = 1
    cnt, edges = np.histogram(k, bins=nbins)
    mid = (edges[:-1] + edges[1:]) / 2
    ok = cnt > 0
    res = stats.linregress(np.log10(mid[ok]), np.log10(cnt[ok] / cnt.sum()))
    return -np.sign(res.slope) * res.rvalue ** 2, k.mean()

X, true_lab, true_load, trait = simulate()
R = np.corrcoef(X.T)                                        # 基因 × 基因相关矩阵
betas = list(range(1, 13))

In [ ]:
fits, mean_k = [], []
for b in betas:
    A = np.abs(R) ** b
    f, mk = scale_free_fit(A)
    fits.append(f); mean_k.append(mk)
beta_star = next(b for b, f in zip(betas, fits) if f >= 0.85)
print("选定 beta =", beta_star)
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
ax[0].plot(betas, fits, "o-"); ax[0].axhline(0.85, color="r", ls="--")
ax[0].set_xlabel("beta"); ax[0].set_ylabel("无标度拟合指数")
ax[1].plot(betas, mean_k, "o-"); ax[1].set_xlabel("beta"); ax[1].set_ylabel("平均连接度")
plt.tight_layout()

In [ ]:
# 核对（不用修改）
assert fits is not None and len(fits) == 12, "fits 应当有 12 个值，依次对应 beta = 1…12"
_ref = [scale_free_fit(np.abs(R) ** b)[0] for b in betas]
assert np.allclose(fits, _ref, atol=1e-6), "拟合指数与参考值不符：邻接应为 np.abs(R) ** b（无符号），并调用 scale_free_fit"
assert beta_star == next(b for b, f in zip(betas, _ref) if f >= 0.85), "beta_star 应是拟合指数首次 >= 0.85 的最小 beta，不是拟合指数最大的 beta"
print("✓ 任务 1 通过")

## 任务 2：拓扑重叠与层次聚类

用任务 1 选出的 $\beta$ 构建邻接矩阵（对角线置 0），按
$\mathrm{TOM}_{ij}=\dfrac{l_{ij}+a_{ij}}{\min(k_i,k_j)+1-a_{ij}}$，$l=AA$
计算拓扑重叠（对角线置 1）。再以 $1-\mathrm{TOM}$ 为距离做平均连锁层次聚类，在高度 0.93 处切树，少于 10 个基因的簇记为 0（相当于 WGCNA 的灰色“未归类”）。

要得到的变量：`T`（120 × 120 的 TOM 矩阵）、`clusters`（长度 120 的整数数组，0 为未归类）。

期望看到：4 个真实模块基本被完整找回，噪声基因大多落在 0 类。
思考：把切树高度改成 0.98、最小模块改成 5，结果怎样变？

In [ ]:
# 准备（直接运行）
def cut_modules(Z, height=0.93, min_size=10):
    """按高度切树；小于 min_size 的簇记为 0；其余按大小从大到小编号 1, 2, ..."""
    c = fcluster(Z, t=height, criterion="distance")
    ids, counts = np.unique(c, return_counts=True)
    big = [i for i, m in sorted(zip(ids, counts), key=lambda x: -x[1]) if m >= min_size]
    return np.array([big.index(x) + 1 if x in big else 0 for x in c])

In [ ]:
A = np.abs(R) ** beta_star
np.fill_diagonal(A, 0.0)
k = A.sum(axis=1)
T = (A @ A + A) / (np.minimum.outer(k, k) + 1 - A)
np.fill_diagonal(T, 1.0)
dist = 1.0 - T
np.fill_diagonal(dist, 0.0)
tree = linkage(squareform(dist, checks=False), method="average")
clusters = cut_modules(tree)
print("找到 %d 个模块；列联表（行 = 真实标签 0..4，列 = 聚类 0..%d）：" % (clusters.max(), clusters.max()))
for t in range(5):
    print(t, [int(np.sum((true_lab == t) & (clusters == c))) for c in range(clusters.max() + 1)])

In [ ]:
# 核对（不用修改）
assert T is not None and T.shape == (120, 120), "T 应是 120 × 120 的矩阵"
assert np.allclose(T, T.T), "TOM 应当对称"
assert np.allclose(np.diag(T), 1.0), "TOM 的对角线应置为 1"
assert T.min() >= 0 and T.max() <= 1 + 1e-9, "TOM 应在 [0, 1] 内；检查分母是否写成 min(k_i, k_j) + 1 - a_ij"
assert clusters is not None and len(clusters) == 120, "clusters 应有 120 个标签"
for m in range(1, 5):
    lab = clusters[true_lab == m]
    top = np.bincount(lab[lab > 0]).max() if (lab > 0).any() else 0
    assert top >= 0.8 * len(lab), f"真实模块 {m} 应基本落在同一个聚类中；检查距离是否用了 1 - TOM、方法是否为 average"
assert (clusters[true_lab == 0] == 0).mean() >= 0.6, "噪声基因大多应为未归类（0）；检查切树高度 0.93 与最小模块 10"
print("✓ 任务 2 通过")

## 任务 3：模块特征基因、模块–性状关联与 hub 基因

对真实模块 1 的基因（`true_lab == 1`），用标准化表达的第一主成分作为模块特征基因 ME（`eigengene` 已写好，符号与模块平均表达一致）；计算 ME 与性状的相关 `r_me`；再计算每个基因的模块成员度 $k_\mathrm{ME}=\mathrm{cor}(x_i,\mathrm{ME})$，取 kME 最高的 3 个基因作为 hub。

要得到的变量：`r_me`（浮点数）、`kme1`（模块 1 每个基因的 kME，顺序与 `idx1` 一致）、`hub1`（3 个基因的列下标）。

期望看到：模块 1 与性状显著相关；hub 的真实载荷明显高于模块平均。
思考：模块 1 的 hub 是“驱动性状的基因”吗？在本模拟中，所有模块 1 基因都只是同一潜在因子的读数，hub 只是噪声最小的读数。这就是“hub 适合作标志物、不等于驱动者”（见 [解读共表达模块](gene-network-inference.module-interpretation.html)）。

In [ ]:
# 准备（直接运行）
Xz = (X - X.mean(0)) / X.std(0)
idx1 = np.where(true_lab == 1)[0]

def eigengene(Xm):
    U, S, Vt = np.linalg.svd(Xm, full_matrices=False)
    me = U[:, 0] * S[0]
    return me if np.corrcoef(me, Xm.mean(axis=1))[0, 1] > 0 else -me

me1 = eigengene(Xz[:, idx1])

In [ ]:
r_me = stats.pearsonr(me1, trait)[0]
kme1 = np.array([np.corrcoef(Xz[:, g], me1)[0, 1] for g in idx1])
hub1 = idx1[np.argsort(-kme1)[:3]]
print("ME–性状 r = %.2f；hub = %s；hub 载荷 = %s；模块平均载荷 = %.2f"
      % (r_me, hub1.tolist(), np.round(true_load[hub1], 2).tolist(), true_load[idx1].mean()))

In [ ]:
# 核对（不用修改）
assert r_me is not None and abs(r_me) > 0.5, "模块 1 的 ME 应与性状强相关（|r| > 0.5）；检查是否用了 me1 与 trait"
assert kme1 is not None and len(kme1) == len(idx1), "kme1 的长度应等于模块 1 的基因数"
assert np.all(np.abs(kme1) <= 1 + 1e-9), "kME 是相关系数，应在 [-1, 1] 内"
assert hub1 is not None and len(hub1) == 3 and set(np.asarray(hub1)) <= set(idx1), "hub1 应是 3 个模块 1 基因的列下标（用 idx1[...] 换算，而不是 kme1 内部的位置）"
assert true_load[np.asarray(hub1)].mean() > true_load[idx1].mean(), "hub 的真实载荷应高于模块平均；检查是否按 kME 从大到小排序"
print("✓ 任务 3 通过")

## 任务 4：相关与偏相关：区分直接与间接关联

模拟一条强调控链 A → B → C（A 不直接作用于 C）、一条弱的直接调控 D → E，外加 3 个无关基因，共 500 个样本。分别计算 A 与 C 的皮尔逊相关 `r_AC` 和偏相关 `pr_AC`（控制其余全部基因）。偏相关由精度矩阵 $\Omega=\Sigma^{-1}$ 换算：$\rho_{ij}=-\omega_{ij}/\sqrt{\omega_{ii}\omega_{jj}}$。这里样本远多于基因，可以直接求逆；样本少于基因时需要收缩，见 [偏相关与收缩估计](gene-network-inference.partial-correlation-shrinkage.html)。

要得到的变量：`r_AC`、`pr_AC`（浮点数）、`pcor`（8 × 8 偏相关矩阵）。

期望看到：A 与 C 的相关很高（约 0.9，比弱的直接边 D–E 还高），偏相关接近 0。

In [ ]:
# 准备（直接运行）
rng4 = np.random.default_rng(4)
nn = 500
a_ = rng4.normal(size=nn)
b_ = 0.95 * a_ + np.sqrt(1 - 0.95 ** 2) * rng4.normal(size=nn)
c_ = 0.95 * b_ + np.sqrt(1 - 0.95 ** 2) * rng4.normal(size=nn)
d_ = rng4.normal(size=nn)
e_ = 0.5 * d_ + np.sqrt(1 - 0.5 ** 2) * rng4.normal(size=nn)
Y = np.column_stack([a_, b_, c_, d_, e_] + [rng4.normal(size=nn) for _ in range(3)])   # 列 0–4 = A–E

In [ ]:
Rc = np.corrcoef(Y.T)
r_AC = Rc[0, 2]
Om = np.linalg.inv(Rc)
dd = np.sqrt(np.diag(Om))
pcor = -Om / np.outer(dd, dd)
np.fill_diagonal(pcor, 1.0)
pr_AC = pcor[0, 2]
print("r(A, C) = %.2f，偏相关 = %.3f；r(A, B) = %.2f，偏相关 = %.2f；r(D, E) = %.2f，偏相关 = %.2f"
      % (r_AC, pr_AC, Rc[0, 1], pcor[0, 1], Rc[3, 4], pcor[3, 4]))

In [ ]:
# 核对（不用修改）
assert r_AC is not None and r_AC > 0.7, "A 与 C 的普通相关应很高（约 0.9）；检查是否取了 [0, 2]"
assert pcor is not None and np.asarray(pcor).shape == (8, 8), "pcor 应是 8 × 8 矩阵"
assert pr_AC is not None and abs(pr_AC) < 0.1, "控制 B 后 A 与 C 的偏相关应接近 0；检查公式前面的负号和分母 sqrt(ω_ii ω_jj)"
assert pcor[0, 1] > 0.4 and pcor[1, 2] > 0.4, "A–B、B–C 是直接边，偏相关应保持较大的正值"
print("✓ 任务 4 通过")

## 任务 5：评估推断结果：AUPR 与早期精度比

写两个函数：`aupr(score, y)` 按“每找到一条真边时的精确率取平均”计算 AUPR；`epr(score, y)` 计算早期精度比：前 $k$ 个预测（$k$ 为真边数）中真边的比例，除以真边比例 $\pi$。`score` 越大越可能是真边，`y` 是 0/1 数组。然后在任务 4 的数据上比较 $|r|$ 与 $|$偏相关$|$ 两种打分：真边是 A–B、B–C、D–E 三条，候选对是 8 个基因的全部 28 对。

要得到的变量：函数 `aupr`、`epr`，以及 `aupr_r`、`aupr_p`（两种打分的 AUPR）。

期望看到：手算例子“真、假、真、假、假、真、假、假、假、假”的 AUPR 约 0.72（见 [推断结果的评估](gene-network-inference.evaluation.html)）；偏相关的 AUPR 为 1；相关打分把间接边 A–C 排在弱真边 D–E 之前，AUPR 约 0.92。

> **提示：** `order = np.argsort(-score, kind="stable")`；精确率 = 累计真边数 / 名次。

In [ ]:
# 准备（直接运行）
iu8 = np.triu_indices(8, 1)
truth = np.zeros((8, 8), dtype=int)
truth[0, 1] = truth[1, 2] = truth[3, 4] = 1
y8 = truth[iu8]

In [ ]:
def aupr(score, y):
    y = np.asarray(y)[np.argsort(-np.asarray(score), kind="stable")]
    prec = np.cumsum(y) / np.arange(1, len(y) + 1)
    return prec[y == 1].mean()

def epr(score, y):
    y = np.asarray(y)
    k = int(y.sum())
    top = y[np.argsort(-np.asarray(score), kind="stable")[:k]]
    return top.mean() / y.mean()

Rc = np.corrcoef(Y.T)
aupr_r = aupr(np.abs(Rc)[iu8], y8)
aupr_p = aupr(np.abs(pcor)[iu8], y8)
print("AUPR：|r| = %.3f，|偏相关| = %.3f；随机 = %.3f" % (aupr_r, aupr_p, y8.mean()))

In [ ]:
# 核对（不用修改）
_s = np.arange(10, 0, -1)
_y = np.array([1, 0, 1, 0, 0, 1, 0, 0, 0, 0])
assert aupr(_s, _y) is not None and abs(aupr(_s, _y) - (1 + 2 / 3 + 3 / 6) / 3) < 1e-6, "手算例子的 AUPR 应为 (1 + 2/3 + 3/6)/3 ≈ 0.722；精确率要用“累计真边数 / 当前名次”"
assert abs(epr(_s, _y) - (2 / 3) / 0.3) < 1e-6, "手算例子前 3 名中有 2 条真边，EPR = (2/3)/0.3 ≈ 2.22；检查 k 是否取真边数、是否除以 π"
assert abs(aupr_p - 1.0) < 1e-9, "偏相关应把三条真边排在最前，AUPR = 1；检查是否对 pcor 取了绝对值"
assert aupr_r < 0.99, "普通相关会把间接边 A–C 排在弱真边 D–E 之前，AUPR 应低于 1；检查是否用了 np.abs(Rc)[iu8]"
print("✓ 任务 5 通过")

## 任务 6：开放小分析：乳腺癌活检的 ER 相关模块

数据：`data/breast_gds4057_expr_top1000.csv`（1000 个高变基因 × 103 例 HER2 阴性乳腺癌化疗前活检，GEO GDS4057）与 `data/breast_gds4057_samples.csv`（ER 免疫组化状态等），来源见 [真实数据案例](gene-network-inference.html)。

自己决定：有符号还是无符号网络、$\beta$、切树高度与最小模块大小。完成后找出与 ER 状态（`er_ihc` 为 `+`/`-`，`NA` 剔除）相关最强的模块，报告它的大小、ME 与 ER 的相关 `best_r`，以及 kME 最高的若干个 hub 基因名 `hubs`。

要得到的变量：`n_modules`（找到的模块数，整数）、`best_r`（浮点数）、`hubs`（至少 3 个基因名的列表）。

思考：这些 hub 基因提示该模块反映的是什么生物学过程或细胞组成？换一个 $\beta$ 或切树高度，结论是否改变？

In [ ]:
# 准备（直接运行）
import pandas as pd
expr = pd.read_csv("data/breast_gds4057_expr_top1000.csv", index_col="gene")
samples = pd.read_csv("data/breast_gds4057_samples.csv", index_col="sample")
Xb = expr.T.loc[samples.index]
Zb = (Xb - Xb.mean()) / Xb.std(ddof=0)
gnames = np.array(Zb.columns)

In [ ]:
Rb = np.corrcoef(Zb.values.T)
Ab = ((1 + Rb) / 2) ** 6                        # 有符号网络，beta = 6
np.fill_diagonal(Ab, 0.0)
kb = Ab.sum(1)
Tb = (Ab @ Ab + Ab) / (np.minimum.outer(kb, kb) + 1 - Ab)
np.fill_diagonal(Tb, 1.0)
Db = 1 - Tb
np.fill_diagonal(Db, 0.0)
treeb = linkage(squareform(Db, checks=False), method="average")
modb = cut_modules(treeb, height=0.85, min_size=20)
n_modules = int(modb.max())
er = samples["er_ihc"].map({"+": 1, "-": 0})
ok = er.notna().values
best_r, best_m = 0.0, None
for m in range(1, n_modules + 1):
    me = eigengene(Zb.values[:, modb == m])
    r = np.corrcoef(me[ok], er.values[ok])[0, 1]
    if abs(r) > abs(best_r):
        best_r, best_m, best_me = r, m, me
idx = np.where(modb == best_m)[0]
kme = np.array([np.corrcoef(Zb.values[:, i], best_me)[0, 1] for i in idx])
hubs = list(gnames[idx][np.argsort(-kme)[:6]])
print("模块数 %d；与 ER 最相关的模块 %d（%d 个基因），r = %.2f；hub：%s" % (n_modules, best_m, len(idx), best_r, ", ".join(hubs)))

In [ ]:
# 核对（不用修改）
assert isinstance(n_modules, (int, np.integer)) and n_modules >= 1, "n_modules 应是正整数"
assert best_r is not None and -1 <= float(best_r) <= 1, "best_r 是相关系数，应在 [-1, 1] 内"
assert hubs is not None and len(hubs) >= 3 and set(hubs) <= set(gnames), "hubs 应是至少 3 个数据中的基因名"
print("✓ 任务 6 通过")